# ENVRI HUB library exercise
In this notebook you'll learn how to use the *VRE-Lib* component developed by WP 13 and WP 14.
The VRE-Lib is available as a *Python package* on the [public package index](https://pypi.org/project/envrihub/) un the *envrihub* name.
This means it can be installed with:

In [1]:
! pip install --upgrade envrihub
! pip install openapi-spec-validator

The main element you have to import is the _Hub_ object:

In [2]:
from envrihub import Hub

hub = Hub()

## Exercise
Use the _Hub_ object to query the Catalogue of Services and retrieve data for a specific Essential Variable: **Ocean Temperature**. Remember you can always ask the [Environmental Expert](https://chat.envri.eu/) for help. The steps of the exercise are:
+ find a service providing data for Ocean Temperature;
+ access the service and retrieve data;
+ display data on a table and/or a graph.

In [3]:
for res in hub.search_catalogue('ocean temperature'):
    print(res.title)
    print(f'\t resource id: {res.uid}')
    print(f'\t resource type: {res.type}')
    print(f'\t resurce description: {res.description}\n\n')

Argo - EXV17/EXV18 - surface ocean temperature and sub-surface ocean temperature
	 resource id: file:///ArgoFloats_distribution_003
	 resource type: WEB_SERVICE
	 resurce description: ERDDAP data access service to Argo Float surface ocean temperature and sub-surface ocean temperature. The service is parameterized so that only data with good values and good spatio-temporal coordinates are collected and shown on the map. Temperature parameter corresponds to https://vocab.nerc.ac.uk/collection/EXV/current/EXV017/ and https://vocab.nerc.ac.uk/collection/EXV/current/EXV018/ (The EXV table is a programming implementation of Essential Variables including GCOS-defined Essential Climate Variables - https://gcos.wmo.int/site/global-climate-observing-system-gcos/essential-climate-variables/ - and of GOOS-defined Essential Ocean Variables - https://goosocean.org/what-we-do/framework/essential-ocean-variables/).




In [4]:
service_id = 'file:///ArgoFloats_distribution_003'

res = hub.fetch_from_catalogue(service_id)
print(res.title)

Argo - EXV17/EXV18 - surface ocean temperature and sub-surface ocean temperature


Where do you get IDs? Either browsing the catlogue on your own, or with the `search_catalogue` method.

What we have now inside the `res` variable is a `Distribution` object that has the following attributes:
+ `title`: the resource title/display name
+ `uid`: the unique internal identifier to fetch it a later time
+ `description`: a human readable minimal description
+ `type`: whether it is a web service or a file
+ `href`: a link to more metadata
+ `service_documentation`: a link to some material that should allow you to get a hold on how to use the resouce.
+ `metadata`: all the metadata avaiable.

Plus the `is_downloadable` function that answers the fundamental question for all you digital kleptomaniacs: *can I download it on my laptop?*

In [5]:
res.metadata

{'DOI': ['10.17882/42182'],
 'availableFormats': [{'format': 'application/epos.geo+json',
   'href': 'https://catalogue.envri.eu/api/v1/execute/3def00e1-5634-480d-ba02-6bff90c642f5?format=application/geo+json',
   'label': 'GEOJSON',
   'originalFormat': 'application/geo+json',
   'type': 'ORIGINAL'}],
 'categories': {'children': [{'children': [{'ddss': 'category:ENVRI_conc',
      'name': 'Services'}],
    'code': 'ENVRIServices',
    'color': '#A5EBA9',
    'id': '1',
    'imgUrl': 'https://gitlab.a.incd.pt/envri-hub-next/cos-icons/-/raw/main/services.png',
    'linkUrl': '',
    'name': 'Services'}],
  'name': 'domains'},
 'dataProvider': [{'country': 'FR',
   'dataProviderLegalName': 'Euro-Argo ERIC',
   'dataProviderUrl': 'https://www.euro-argo.eu/',
   'instanceid': 'ded6fbf2-d357-49fc-951d-a0c4599e7826',
   'metaid': 'ded6fbf2-d357-49fc-951d-a0c4599e7826',
   'uid': 'ded6fbf2-d357-49fc-951d-a0c4599e7826'}],
 'description': 'ERDDAP data access service to Argo Float surface ocean 

Quite some information isn't it?
That's becasue you might want to know what you'll find inside the data *before* opening it. You know... To avoid jumpscares.

Now it's finally time to access the *actual data* with the `dao` attribute, that *always* comes with helpful documentation you can access with the *help()* function or with third party Jupyter extensions.

In [6]:
catalogue_dao = res.dao
help(catalogue_dao)

Help on ArgoTEMPAdjustedERDDAP in module abc object:

class ArgoTEMPAdjustedERDDAP(envrihub.data_access.argo.ArgoERDDAPDao)
 |  Method resolution order:
 |      ArgoTEMPAdjustedERDDAP
 |      envrihub.data_access.argo.ArgoERDDAPDao
 |      envrihub.data_access.models.DataAccessObject
 |      ABC
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  access(
 |      self,
 |      /,
 |      *,
 |      time_less='2025-02-01T00:00:00Z',
 |      pres_less='10',
 |      longitude_greater='-21',
 |      longitude_less='30',
 |      latitude_less='66',
 |      pres_greater='0',
 |      time_greater='2025-01-01T00:00:00Z',
 |      latitude_greater='29',
 |      additional_headers: dict = None
 |  ) from envrihub.data_access.catalogue_metadata_client.build_access_function.<locals>
 |      ERDDAP data access service to Argo Float surface ocean temperature and sub-surface ocean temperature. The service is parameterized so that only data with good values and good spatio-temporal coordinates a

And now let's do some magic with data. The [Environmental Expert](https://chat.envri.eu/) can help you with this.

In [7]:
import pandas as pd
import json

service_response = catalogue_dao.access(latitude_less='33', latitude_more='29', longitude_less='30', longitude_more='-21', pres_less='10', time_less='2025-02-01T00:00:00Z', pres_more='0', time_more='2025-01-01T00:00:00Z')

# 1. The service returns a geoJSON, let's load it
data = json.loads(service_response)

# 2. Extract features
features = data['features']

# 3. Transform 'properties' into dictionaries

rows = []
for f in features:
    # Get all properties (temp, pres, psal, ecc.)
    record = f['properties'].copy()
    
    # Extract coordinates: [longitudine, latitudine]
    record['longitude'] = f['geometry']['coordinates'][0]
    record['latitude'] = f['geometry']['coordinates'][1]
    
    rows.append(record)

# 4. Create DataFrame
df = pd.DataFrame(rows)

# 5. Remove empty column
df = df.replace("", float('nan'))
df_pulito = df.dropna(axis=1, how='all')

# Print result
print(f"Clean table: {df_pulito.shape[0]} rows and {df_pulito.shape[1]} columns.")
print(df_pulito.head())

Clean table: 146 rows and 47 columns.
  fileNumber     data_type format_version handbook_version  \
0    2903443  Argo profile            3.1              1.2   
1    2903443  Argo profile            3.1              1.2   
2    2903443  Argo profile            3.1              1.2   
3    2903443  Argo profile            3.1              1.2   
4    2903443  Argo profile            3.1              1.2   

    reference_date_time         date_creation           date_update  \
0  1950-01-01T00:00:00Z  2024-06-06T15:48:07Z  2026-05-04T00:22:28Z   
1  1950-01-01T00:00:00Z  2024-06-06T15:48:07Z  2026-05-04T00:22:28Z   
2  1950-01-01T00:00:00Z  2024-06-06T15:48:07Z  2026-05-04T00:22:28Z   
3  1950-01-01T00:00:00Z  2024-06-06T15:48:07Z  2026-05-04T00:22:28Z   
4  1950-01-01T00:00:00Z  2024-06-06T15:48:07Z  2026-05-04T00:22:28Z   

  platform_number project_name                                      pi_name  \
0         2903443    Argo WHOI  SUSAN WIJFFELS, STEVEN JAYNE, PELLE ROBBINS   
1   